# Chain of Thought & ReAct — Beginner Lesson (Python)

This notebook teaches two prompting techniques used to make LLMs reason better:

1. **Chain of Thought (CoT)** — getting the model to think step-by-step in text
2. **ReAct (Reason + Act)** — getting the model to think AND use real tools

We'll also cover **Python functions**: what they are, why we use them, and how they fit into building these patterns.

No prior LLM experience needed. Basic Python helps but isn't required — every code cell is explained.

## 1. What is Chain of Thought (CoT)?

Normally, if you ask an LLM a question, it jumps straight to an answer. For simple things that's fine. For multi-step problems (math, logic, planning), jumping straight to an answer often produces mistakes.

**Chain of Thought** is a prompting technique where you ask the model to reason step-by-step *before* giving a final answer — instead of guessing immediately.

Example:

**Without CoT:**
> Q: If a train travels 60 km in 1.5 hours, what's its speed in km/h?
> A: 40 km/h ❌ (wrong, but stated confidently)

**With CoT:**
> Q: If a train travels 60 km in 1.5 hours, what's its speed in km/h? Let's think step by step.
> A: Speed = distance / time = 60 / 1.5 = 40 km/h ✅

Same question — but asking it to "think step by step" makes it show its work, which usually makes the answer more reliable.

**Key point:** CoT is just a *prompt*. The model is still only doing text prediction — it's not calling any real tools. It's "thinking out loud," nothing more.

## 2. What is ReAct?

CoT helps the model reason, but it can still get facts or math wrong because it's reasoning purely from memory.

**ReAct = Reason + Act.** The model alternates between:

- **Thought** — reasoning about what to do next
- **Action** — calling a real tool (calculator, search, database, etc.)
- **Observation** — the actual result from that tool

It repeats this loop until it has enough to give a **Final Answer**.

```
Thought: I need to calculate this.
Action: calculator(45*12+7)
Observation: 547
Thought: I now have the answer.
Final Answer: 547
```

The difference from CoT: the "Observation" is a **real, verified result** from code you actually run — not something the model made up.

## 3. CoT vs ReAct — the key difference

| | Chain of Thought | ReAct |
|---|---|---|
| What it does | Reasons in text | Reasons **and** takes real actions |
| Uses real tools? | No | Yes |
| Good for | Logic/math the model can do in its "head" | Anything needing live data or exact computation |
| Risk | Can still hallucinate | Grounded by real tool results |

**Rule of thumb:** use CoT when reasoning alone is enough. Use ReAct when the model needs facts or precision it can't reliably produce on its own.

## 4. Setup

We'll call OpenAI's API. Install the `requests` library first (skip if already installed).

In [ ]:
!pip install requests

Set your API key below. Never share this key or commit it to public code.

In [ ]:
api_key = "YOUR_API_KEY"  # replace with your real key

## 5. Python functions — what and why

Before building anything, we need **functions**.

A function is a reusable block of code with a name. You define it once, then *call* it as many times as you want instead of retyping the same code.

```python
def add(a, b):
    return a + b

add(2, 3)   # 5
add(10, 5)  # 15
```

- `def` starts a function definition
- `add` is the function's name
- `(a, b)` are the **parameters** — inputs the function expects
- `return` sends a value back to whoever called the function

**Why we use functions here:**
- We'll be calling the LLM API many times — wrapping that in a function avoids repeating the same `requests.post(...)` code
- We'll need a calculator that the model can "call" as a tool — that only works if it's a clean, reusable function
- Functions make the ReAct loop readable: `call_llm(...)`, `calculator(...)` instead of a wall of repeated code

**When to reach for a function:** any time you're about to copy-paste code you already wrote, or any time a piece of logic (like "send this to the API") needs a clear name and reusable shape.

In [ ]:
# quick practice: a function with one job
def add(a, b):
    return a + b

print(add(2, 3))
print(add(10, 5))

## 6. Wrapping the API call in a function

Right now, calling OpenAI means writing the same `requests.post(...)` block every time. Let's turn it into a function so we just call `call_llm(messages)`.

In [ ]:
import requests

def call_llm(messages, model="gpt-4o-mini"):
    res = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={"model": model, "messages": messages}
    )
    return res.json()["choices"][0]["message"]["content"]

Now every future API call is just one line: `call_llm(messages)`. This is the payoff of using a function — the messy request/response handling is hidden away, and we call it just like `add(2, 3)` above.

## 7. Chain of Thought in practice

To use CoT, we just add reasoning instructions to the prompt — no extra code needed, no tools, just better prompting.

In [ ]:
messages = [
    {"role": "system", "content": "Think step by step before giving your final answer."},
    {"role": "user", "content": "A shirt costs $40. It's on sale for 25% off. What's the final price?"}
]

print(call_llm(messages))

Notice: this only used our `call_llm` function — no calculator, no tool. The model reasons entirely in text. That's the whole idea of CoT: better prompting, same underlying process.

## 8. Building ReAct — the calculator tool

For ReAct, the model needs a **real** action it can trigger. We'll give it a calculator.

This means writing a function the model's output can trigger — not just a function we call ourselves.

In [ ]:
def calculator(expression):
    return eval(expression)

# quick test
print(calculator("45*12+7"))

⚠️ **Note:** `eval()` runs any Python code passed to it — fine for learning/testing, but unsafe with untrusted input in a real app. A safer version would only allow numbers and math operators.

## 9. Reading the model's requested action

The model will respond with text like `Action: calculator(45*12+7)`. We need to **parse** that text to actually run the calculator — the model can't run code itself, it only writes text.

In [ ]:
import re

def extract_action(reply):
    match = re.search(r"calculator\((.*?)\)", reply)
    if match:
        return match.group(1)   # the expression inside the parentheses
    return None

# quick test
print(extract_action("Thought: I need to multiply.\nAction: calculator(6*7)"))

## 10. Putting it together — the full ReAct loop

Now we combine everything: `call_llm`, `calculator`, and `extract_action` into one loop.

1. Ask the model the question
2. Check if it requested an action
3. If yes: run the real calculator, send the result back as an "Observation"
4. Ask again — now it should give a Final Answer

In [ ]:
def react_solve(question):
    system_prompt = (
        "You run in a loop of Thought, Action, Observation.\n"
        "Available action: calculator(expression)\n"
        "When ready, respond with Final Answer."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    reply = call_llm(messages)
    print(reply)

    expression = extract_action(reply)
    if expression:
        result = calculator(expression)
        print(f"Observation: {result}")

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Observation: {result}"})

        final_reply = call_llm(messages)
        print(final_reply)

In [ ]:
react_solve("What is 45 * 12 + 7?")

## Summary

- **Chain of Thought** = better prompting. Ask the model to reason step by step. No tools, no extra code — just text in, text out.
- **ReAct** = reasoning + real tool calls. The model's text output (`Action: ...`) is parsed by your code, which runs a real function and feeds the true result back in.
- **Functions** are what make ReAct possible: `call_llm()` to talk to the model, `calculator()` as the real tool, `extract_action()` to read what the model asked for. Each does one job, so the loop stays readable.

### Try it yourself
1. Change `react_solve` to handle a second tool, e.g. `def word_count(text): return len(text.split())`
2. Update `extract_action` to detect which tool was requested (calculator vs word_count)
3. Test with a question that needs the new tool

That's the core pattern every LLM "agent" framework builds on.